## Computer Vision on Acadia Footage

#### Date: 1/06/2026
#### Author: Nineveh O'Connell

Go Pro footage from Acadia Confusion Corner

In [1]:
#import libraries
import re
from datetime import datetime, timezone
import pandas as pd
import numpy as np

import time
from pathlib import Path
import glob
import os

import cv2
import yt_dlp
from ultralytics import YOLO
from collections import defaultdict
import supervision as sv
from bs4 import BeautifulSoup
import requests
from IPython.display import display, Image
from PIL import Image as Img
from PIL import ImageTk
from urllib.parse import urljoin


## YOLO modeling of video

In [2]:
# Example usage (for current directory):
p_dir_base = "C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/4- Data Collection/Video Data/8.27.25 Bike Handlebars Jordan Pond GoProMax 09714/"
all_files_pathlib = list(Path(p_dir_base).glob('*.LRV'))



In [3]:
# Load the YOLO model
model = YOLO('yolo11l.pt')

class_list = model.names 

In [4]:

def get_video_properties_cv2(filename):
    video = cv2.VideoCapture(filename)
    
    if not video.isOpened():
        print(f"Error opening video file: {filename}")
        return None

    # Get dimensions
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Get length
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps > 0:
        duration_seconds = frame_count / fps
    else:
        duration_seconds = 0

    video.release()

    return {
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds
    }

# Example Usage
properties = get_video_properties_cv2(all_files_pathlib[0])
if properties:
    print(f"Dimensions: {properties['width']}x{properties['height']}")
    print(f"Length (seconds): {properties['duration_seconds']:.2f}s")


Dimensions: 1408x704
Length (seconds): 482.48s


In [5]:
# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
# this will define that function
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

In [ ]:
for video_path in all_files_pathlib:

    video_base_name = os.path.basename(video_path)
    out_csv_path = f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ConfusionCorner/cv_output{video_base_name}.csv"

    # Open video
    cap = cv2.VideoCapture(video_path)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    curr_frame_num = 0

    # name window for viewer
    window_name = "Fullscreen Video"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    # to save results
    resultsList = []

    # the approach of using a while looks and checking for success isn't working
    # instead let's use a different while condition
    # this was the condition before: while cap.isOpened()

    while curr_frame_num < n_frames_total:
        ret, frame = cap.read()

        # Get current frame number
        frame_num = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        curr_frame_num = frame_num
        
        # If detections occur, do operations for every tenth frame 
        if frame_num % 10 == 0:

            if not ret:
                print("this is the for-loop internal call of")
                print("Video completed or error reading frame.")
                break

            # Process frame for detections
            results = model.track(frame, classes = [0,1,2,3,5,7,11], persist = True)

            if len(results) > 0:

                timestamp = frame_num / frame_rate
                cv2.putText(frame, f'Timestamp: {timestamp:.2f}s', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                            1, (255, 255, 255), 2, cv2.LINE_AA)
            
                # Here you can save the frame or timestamp if needed
                for r in results:

                    if r.boxes.id is not None:
                        boxes = r.boxes.xyxy.cpu()  # Boxes object for bbox outputs
                        print(boxes)
                        track_ids = r.boxes.id.int().cpu().tolist()
                        class_indices = r.boxes.cls.int().cpu().tolist()
                        confidences = r.boxes.conf.cpu()

                        # Loop through each detected object
                        for box, track_id, class_idx, conf in zip(boxes, track_ids, class_indices, confidences):
                            x1, y1, x2, y2 = map(int, box)
                            cx = (x1 + x2) // 2  # Calculate the center point
                            cy = (y1 + y2) // 2            

                            class_name = class_list[class_idx]

                            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                            
                            cv2.putText(frame, f"ID: {track_id} {class_name}", (x1, y1 - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
                            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) 

                            dfKeyFeatures = pd.DataFrame({'id' : [track_id], 
                                                        'class' : [class_name], 
                                                        'confidence' : [conf], 
                                                        'cx' : [cx], 
                                                        'cy' : [cy], 
                                                        'bottomLeftx' : [x1],
                                                        'bottomLefty' : [y1],
                                                        'upperRightx' : [x2],
                                                        'upperRighty' : [y2],
                                                        'timestamp' : [timestamp]})
                            resultsList.append(dfKeyFeatures)

                # Display the frame
                #cv2.rectangle(frame, (1550, 950), (1950, 1250), (100, 80, 250), 2)

                cv2.imshow(window_name, frame)
            
        # if video has already been analyzed, break and move to next file in path
        if Path(out_csv_path).is_file():
            break

        # if manual override, break and move to next file in path
        if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

    # compile results
    if(len(resultsList) > 1 ):
        combined_df = pd.concat(resultsList)
        combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)
        # export results to csv
        combined_df.to_csv(out_csv_path, index=False) 



0: 320x640 1 person, 423.0ms
Speed: 2.4ms preprocess, 423.0ms inference, 2.0ms postprocess per image at shape (1, 3, 320, 640)
tensor([[ 542.8314,   28.6050, 1402.1067,  679.6996]])

0: 320x640 1 person, 345.8ms
Speed: 1.4ms preprocess, 345.8ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)
tensor([[ 542.0277,   32.4492, 1401.3507,  649.3112]])

0: 320x640 1 person, 353.8ms
Speed: 1.8ms preprocess, 353.8ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)
tensor([[ 550.0784,   32.4243, 1408.0000,  684.8779]])

0: 320x640 1 person, 381.1ms
Speed: 3.1ms preprocess, 381.1ms inference, 1.3ms postprocess per image at shape (1, 3, 320, 640)
tensor([[ 537.5663,   29.6556, 1408.0000,  650.7280]])

0: 320x640 1 person, 387.1ms
Speed: 1.8ms preprocess, 387.1ms inference, 1.6ms postprocess per image at shape (1, 3, 320, 640)
tensor([[ 547.4952,   33.9980, 1403.2489,  680.8492]])

0: 320x640 1 person, 339.9ms
Speed: 2.4ms preprocess, 339.9ms inference, 3.0ms postp

LinAlgError: 1-th leading minor of the array is not positive definite

: 